# Colab LLM Station

A reproducible, dual-engine inference and networking deployment framework designed for Google Colab and cloud GPU environments.

---

## Prerequisites: Tailscale Auth Key Setup (One-Time Configuration)

Google Colab runtimes reside behind network address translation (NAT) without dedicated public IPv4 addresses. An ephemeral Tailscale Auth Key allows headless, automated authentication on runtime startup without requiring browser interactions or QR codes.

### 1. Generate Auth Key in Tailscale Admin Console
1. Open the [Tailscale Admin Console - Keys](https://login.tailscale.com/admin/settings/keys).
2. Click **Generate auth key**.
3. Configure the following key settings:
   - **Description**: Enter an identifier (e.g., `colab-llm-station`).
   - **Reusable**: **Enabled**. Allows the same key to be reused across Colab runtime restarts without generating a new key each session.
   - **Ephemeral**: **Enabled (Crucial)**. Automatically removes the machine from your Tailnet when the Colab instance shuts down, preventing accumulation of offline duplicate nodes.
   - **Pre-authorized**: **Enabled**. Automatically approves the device into your network upon connection.
   - **Tags** (Optional): Assign tags if required by your Tailnet ACL policy (e.g., `tag:server`).
4. Click **Generate key** and copy the generated key (`tskey-auth-...`). Store it temporarily; Tailscale displays this key only once.

### 2. Store Key in Google Colab Secrets
1. In the Google Colab left navigation panel, click the **Secrets** tab (padlock/key icon).
2. Click **Add new secret**.
3. Set the configuration:
   - **Name**: `TAILSCALE_AUTHKEY` (must match exactly, case-sensitive).
   - **Value**: Paste your `tskey-auth-...` key string.
4. Toggle the **Notebook access** switch to **ON** to allow Python runtime access via `google.colab.userdata`.

## Step 1: Mount Google Drive & Establish Tailscale SSH Connection
Mounts persistent Google Drive storage and initializes the Tailscale daemon in userspace networking mode. SSH access becomes available within seconds without waiting for large ML package installations.

In [ ]:
# ==============================================================================
# Standalone SSH Connection Cell (Tailscale SSH: Zero-config, Passwordless)
# ==============================================================================
from google.colab import drive, userdata

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Retrieve Tailscale Auth Key from Secrets
authkey = userdata.get('TAILSCALE_AUTHKEY')

# 3. Connect to Tailscale with Native SSH Server
!cd /content/drive/MyDrive/colab/colab-llm-station && \
 bash llm.sh tunnel tailscale up "$authkey"

print("\n" + "="*60)
print("[OK] Tailscale SSH service is ready.")
print("1. Local Terminal Connection:")
print("   ssh root@colab-llm-station")
print("\n2. Local VS Code (Remote - SSH) Connection:")
print("   Press F1 -> Select Remote-SSH: Connect to Host... -> root@colab-llm-station")
print("="*60)


## Step 2: Environment Provisioning (Optional in Notebook / Available via SSH)
> Note: Installs pinned binary versions of Ollama, vLLM, and system packages.
> This step can be executed here in the notebook, or directly inside your remote SSH / VS Code terminal session.

In [ ]:
%%bash
cd /content/drive/MyDrive/colab/colab-llm-station

# Provision all runtime dependencies (vLLM, Ollama, system packages)
bash setup.sh

# Optional: Provision specific engine only
# bash setup.sh vllm    # vLLM and system dependencies only
# bash setup.sh ollama  # Ollama and system dependencies only


## Step 3: Launch LLM Inference Engine & Expose Service Port
> Note: Launches background inference daemon and maps internal service port to your private Tailnet.

In [ ]:
%%bash
cd /content/drive/MyDrive/colab/colab-llm-station

# Enterprise Tier (A100 / H100): Launch vLLM on Port 8000
bash llm.sh tunnel tailscale serve 8000
bash llm.sh vllm serve Qwen/Qwen2.5-Coder-32B-Instruct

# Standard Tier (T4 / L4): Uncomment to run Ollama on Port 11434
# bash llm.sh tunnel tailscale serve 11434
# bash llm.sh ollama serve


## Step 4: System and Tunnel Status Verification

In [ ]:
!cd /content/drive/MyDrive/colab/colab-llm-station && bash llm.sh status
